#### Run the below in terminal to setup 

```bash
pip install kaggle
gcloud auth application-default login
```

`mkdir -p ~/.kaggle && echo KGAT_a5ff0c573d53c0fd5b70bc868c5940df > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token`

#### Use 'elt' kernel / environment.

#### 1. Download from Kaggle

In [21]:
from pathlib import Path
import kaggle

# Configuration
# kaggle_dataset = 'olistbr/brazilian-ecommerce'
# data_dir = Path("../data/olist")
# kaggle_dataset = 'psparks/instacart-market-basket-analysis'
# data_dir = Path("../data/instacart")

data_dir.mkdir(parents=True, exist_ok=True)

# Authenticate with Kaggle
kaggle.api.authenticate()

# Download and extract the dataset
kaggle.api.dataset_download_files(
    kaggle_dataset,
    path=str(data_dir),
    unzip=True
)

Dataset URL: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce


#### 2. Upload to BigQuery

In [5]:
from google.cloud import bigquery

client = bigquery.Client()
project_id = "project-7781a5d3-3f4e-4cd5-a43"
dataset_id = "olist"  

dataset_ref = bigquery.Dataset(f"{project_id}.{dataset_id}")
dataset_ref.location = "US"

try:
    client.create_dataset(dataset_ref)
    print(f"✅ Dataset {dataset_id} created.")
except Exception:
    print(f"ℹ️ Dataset {dataset_id} already exists.")


ℹ️ Dataset olist already exists.


In [23]:
import pandas as pd

for f in Path("./data/olist").glob("*.csv"):
    table_id = f.stem  # keep Kaggle filename stem
    df = pd.read_csv(f)

    full_table_id = f"{project_id}.{dataset_id}.{table_id}"
    job = client.load_table_from_dataframe(df, full_table_id)
    job.result()
    print(f"✅ Uploaded {f.name} ")


✅ Uploaded olist_geolocation_dataset.csv 
✅ Uploaded olist_products_dataset.csv 
✅ Uploaded olist_order_reviews_dataset.csv 
✅ Uploaded olist_order_items_dataset.csv 
✅ Uploaded olist_orders_dataset.csv 
✅ Uploaded olist_customers_dataset.csv 
✅ Uploaded product_category_name_translation.csv 
✅ Uploaded olist_sellers_dataset.csv 
✅ Uploaded olist_order_payments_dataset.csv 


In [10]:
# Find all CSV files recursively
data_dir = Path("../data/olist")
csv_files = list(data_dir.rglob("*.csv"))
# bypass review file which didn't load properly
#csv_files = list(data_dir.rglob("new_view.csv"))
#csv_files = list(data_dir.rglob("olist_order_reviews_dataset.csv"))
print(csv_files)

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {data_dir}")

print(f"Found {len(csv_files)} CSV file(s)")

[PosixPath('../data/olist/olist_sellers_dataset.csv'), PosixPath('../data/olist/olist_order_ratings_dataset.csv'), PosixPath('../data/olist/product_category_name_translation.csv'), PosixPath('../data/olist/olist_orders_dataset.csv'), PosixPath('../data/olist/olist_order_items_dataset.csv'), PosixPath('../data/olist/olist_customers_dataset.csv'), PosixPath('../data/olist/olist_geolocation_dataset.csv'), PosixPath('../data/olist/olist_order_payments_dataset.csv'), PosixPath('../data/olist/olist_products_dataset.csv')]
Found 9 CSV file(s)


In [11]:
for csv_file in csv_files:
    # Use the filename as the BigQuery table name
    table_name = csv_file.stem.lower()

    # Replace invalid characters in table names
    table_name = "".join(
        character if character.isalnum() or character == "_" else "_"
        for character in table_name
    )

    table_id = f"{project_id}.{bq_dataset}.{table_name}"

    job_config = bigquery.LoadJobConfig(
        autodetect=True,
        skip_leading_rows=1,
        source_format=bigquery.SourceFormat.CSV,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    )

    print(f"Loading {csv_file} into {table_id}")

    with csv_file.open("rb") as source_file:
        load_job = client.load_table_from_file(
            source_file,
            table_id,
            job_config=job_config,
        )

    load_job.result()

    table = client.get_table(table_id)
    print(f"Loaded {table.num_rows} rows into {table_id}")

Loading ../data/olist/olist_sellers_dataset.csv into project-7781a5d3-3f4e-4cd5-a43.brazilian_ecommerce.olist_sellers_dataset
Loaded 3095 rows into project-7781a5d3-3f4e-4cd5-a43.brazilian_ecommerce.olist_sellers_dataset
Loading ../data/olist/olist_order_ratings_dataset.csv into project-7781a5d3-3f4e-4cd5-a43.brazilian_ecommerce.olist_order_ratings_dataset
Loaded 99224 rows into project-7781a5d3-3f4e-4cd5-a43.brazilian_ecommerce.olist_order_ratings_dataset
Loading ../data/olist/product_category_name_translation.csv into project-7781a5d3-3f4e-4cd5-a43.brazilian_ecommerce.product_category_name_translation
Loaded 71 rows into project-7781a5d3-3f4e-4cd5-a43.brazilian_ecommerce.product_category_name_translation
Loading ../data/olist/olist_orders_dataset.csv into project-7781a5d3-3f4e-4cd5-a43.brazilian_ecommerce.olist_orders_dataset
Loaded 99441 rows into project-7781a5d3-3f4e-4cd5-a43.brazilian_ecommerce.olist_orders_dataset
Loading ../data/olist/olist_order_items_dataset.csv into project-